# Análisis de startups LATAM — Clase 02
## Visualizaciones avanzadas: distribuciones, relaciones y correlaciones

En la clase anterior exploramos el dataset y aprendimos a hacer barplots e histogramas.

Hoy subimos un escalón: vamos a usar gráficos que **comparan distribuciones entre grupos**, **muestran relaciones entre dos variables** y **revelan patrones de correlación** entre todas las variables del dataset.

---

**Repaso rápido — ¿qué tipo de gráfico para qué pregunta?**

| Pregunta | Gráfico |
|---|---|
| ¿Cuántos hay por categoría? | Barplot (Clase 01) |
| ¿Cómo se distribuye esta variable? | Histograma (Clase 01) |
| ¿Cómo varía la distribución de X entre grupos? | **Boxplot** (hoy) |
| ¿X e Y están relacionados? ¿Cómo? | **Scatter plot** (hoy) |
| ¿Qué variables se mueven juntas? | **Heatmap de correlación** (hoy) |

> Antes de cada gráfico, siempre preguntarse: **¿qué pregunta de negocio estoy respondiendo?**

## Setup: librerías y datos

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

df = pd.read_csv('startups.csv', index_col=0)
print(f"Dataset cargado: {df.shape[0]} startups, {df.shape[1]} variables")
df.head(3)

---
# Parte 1: Boxplots — comparar distribuciones entre grupos

## ¿Qué muestra un boxplot?

Un boxplot ("diagrama de caja") comprime **toda la distribución de una variable** en 5 números:

```
        |--- bigote superior (hasta 1.5×IQR)
        |
   ┌────┤  ← percentil 75 (Q3)
   │    │
   │ ───┤  ← mediana (Q2, percentil 50)
   │    │
   └────┤  ← percentil 25 (Q1)
        |
        |--- bigote inferior (hasta 1.5×IQR)
        
   ●    ← outlier (fuera de los bigotes)
```

- **La caja** contiene el 50% central de los datos (IQR = Q3 - Q1)
- **La línea del medio** es la mediana
- **Los puntos fuera** son outliers

**Ventaja clave sobre el barplot:** el barplot solo muestra el promedio; el boxplot muestra **toda la forma de la distribución**.

**Pregunta:** ¿cuándo importa ver la distribución completa en lugar de solo el promedio? Piensen en un caso donde dos grupos tienen el mismo promedio pero distribuciones muy distintas.

### `sns.boxplot()` — ARR por etapa de financiamiento

**Pregunta de negocio:** ¿el ARR crece consistentemente con la etapa, o hay mucha variabilidad dentro de cada una?

Ordenamos las etapas de menor a mayor madurez para que la narrativa sea clara.

In [ ]:
orden_etapa = ['Pre-seed', 'Seed', 'Serie A', 'Serie B']

plt.figure(figsize=(10, 5))
sns.boxplot(
    data=df,
    x='etapa',
    y='arr_millones',
    order=orden_etapa,
    palette='Blues'
)
plt.title('Distribución del ARR por Etapa de Financiamiento', fontsize=14)
plt.xlabel('Etapa')
plt.ylabel('ARR (millones USD)')
plt.tight_layout()
plt.show()

### `sns.boxplot()` — Churn mensual por sector

**Pregunta de negocio:** ¿hay sectores con churn estructuralmente más alto? ¿Qué implicaría eso para un inversor?

Ordenamos por mediana de churn para facilitar la comparación.

In [ ]:
orden_sectores_churn = (
    df.groupby('sector_principal')['churn_mensual_pct']
    .median()
    .sort_values(ascending=False)
    .index
)

plt.figure(figsize=(11, 5))
sns.boxplot(
    data=df,
    x='sector_principal',
    y='churn_mensual_pct',
    order=orden_sectores_churn,
    palette='Set2'
)
plt.title('Distribución del Churn Mensual (%) por Sector', fontsize=14)
plt.xlabel('Sector')
plt.ylabel('Churn mensual (%)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

### `sns.boxplot()` — NPS por modelo de negocio

**Pregunta de negocio:** ¿los clientes de startups B2B tienen más o menos satisfacción que los de B2C?

**Pregunta de código:** ¿cómo agregarían una línea horizontal en NPS = 0 para visualizar la frontera entre promotores netos y detractores netos?

In [ ]:
plt.figure(figsize=(9, 5))
sns.boxplot(
    data=df,
    x='modelo_negocio',
    y='nps_score',
    palette='Set3',
    order=['B2B', 'B2C', 'B2B2C']
)
plt.axhline(0, color='red', linestyle='--', linewidth=1, label='NPS = 0')
plt.title('Distribución del NPS Score por Modelo de Negocio', fontsize=14)
plt.xlabel('Modelo de negocio')
plt.ylabel('NPS Score')
plt.legend()
plt.tight_layout()
plt.show()

---
# Parte 2: Scatter plots — relaciones entre dos variables

Un **scatter plot** (gráfico de dispersión) muestra la relación entre **dos variables numéricas**: cada punto es una startup.

Permite ver:
- Si hay correlación positiva / negativa / ninguna
- Si la relación es lineal o no
- Dónde están los outliers
- Si hay grupos o clusters visibles

**Pregunta conceptual:** ¿qué diferencia hay entre *correlación* y *causalidad*? Den un ejemplo con variables del dataset donde podrían confundirse.

### `sns.scatterplot()` — Fondeo vs ARR

**Pregunta de negocio:** ¿las startups que recibieron más fondeo tienen más ARR? ¿La relación es clara o hay mucho ruido?

Usamos `hue='etapa'` para agregar una dimensión extra: ver si las etapas se agrupan visualmente.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df,
    x='fondeo_usd_mm',
    y='arr_millones',
    hue='etapa',
    hue_order=['Pre-seed', 'Seed', 'Serie A', 'Serie B'],
    palette='viridis',
    alpha=0.7,
    s=60
)
plt.title('Fondeo vs ARR — coloreado por Etapa', fontsize=14)
plt.xlabel('Fondeo total (millones USD)')
plt.ylabel('ARR (millones USD)')
plt.legend(title='Etapa')
plt.tight_layout()
plt.show()

### `sns.scatterplot()` — Churn vs NPS

**Pregunta de negocio:** ¿las startups con mejor NPS tienen menor churn? ¿La satisfacción del cliente predice la retención?

Agregamos `size='arr_millones'` para una tercera dimensión: el tamaño del punto representa el ARR.

In [ ]:
df_scatter = df.dropna(subset=['churn_mensual_pct', 'nps_score', 'arr_millones'])

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df_scatter,
    x='nps_score',
    y='churn_mensual_pct',
    hue='sector_principal',
    size='arr_millones',
    sizes=(30, 300),
    alpha=0.7,
    palette='tab10'
)
plt.title('NPS Score vs Churn Mensual — tamaño = ARR', fontsize=14)
plt.xlabel('NPS Score')
plt.ylabel('Churn mensual (%)')
plt.legend(title='Sector', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

### `sns.regplot()` — scatter con línea de tendencia

`sns.regplot()` agrega automáticamente una **línea de regresión lineal** con intervalo de confianza. Útil para ver si la tendencia visual se confirma estadísticamente.

**Pregunta:** ¿qué significa el área sombreada alrededor de la línea? ¿Un área más ancha es buena o mala señal?

In [ ]:
df_reg = df.dropna(subset=['fondeo_usd_mm', 'arr_millones'])

plt.figure(figsize=(9, 5))
sns.regplot(
    data=df_reg,
    x='fondeo_usd_mm',
    y='arr_millones',
    scatter_kws={'alpha': 0.5, 'color': 'steelblue'},
    line_kws={'color': 'red', 'linewidth': 2}
)
plt.title('Fondeo vs ARR — con línea de tendencia', fontsize=14)
plt.xlabel('Fondeo total (millones USD)')
plt.ylabel('ARR (millones USD)')
plt.tight_layout()
plt.show()

---
# Parte 3: Heatmap de correlación — ver todo a la vez

## ¿Qué es la correlación?

La **correlación** mide qué tan relacionadas están dos variables numéricas. Va de **-1 a +1**:

| Valor | Significado |
|---|---|
| `+1.0` | Correlación positiva perfecta: cuando X sube, Y sube igual |
| `+0.5` a `+0.8` | Correlación positiva moderada/fuerte |
| `≈ 0` | Sin correlación lineal aparente |
| `-0.5` a `-0.8` | Correlación negativa moderada/fuerte |
| `-1.0` | Correlación negativa perfecta: cuando X sube, Y baja igual |

El **heatmap de correlación** calcula el coeficiente de correlación de Pearson entre **todas las columnas numéricas** y lo muestra como una grilla coloreada.

> ⚠️ Correlación no implica causalidad. Dos variables pueden correlacionar por pura coincidencia o por una tercera variable oculta.

**Pregunta:** antes de ver el heatmap, ¿qué correlaciones esperan encontrar en el dataset? Hagan 3 predicciones (ej: "creo que fondeo y ARR van a correlacionar positivo").

### `.corr()` + `sns.heatmap()` — matriz de correlación completa

`.corr()` calcula la matriz de correlación. `sns.heatmap()` la visualiza con colores.

- `annot=True` — muestra el número en cada celda
- `cmap='RdYlGn'` — rojo = correlación negativa, verde = correlación positiva
- `center=0` — el centro de la escala de color es 0
- `mask=` — ocultamos el triángulo superior (es espejo del inferior)

In [ ]:
cols_numericas = ['edad_empresa', 'empleados', 'clientes_activos', 'arr_millones',
                   'churn_mensual_pct', 'nps_score', 'fondeo_usd_mm', 'runway_meses']

corr = df[cols_numericas].corr()

# Máscara para el triángulo superior (evitar redundancia)
mask = np.zeros_like(corr, dtype=bool)
mask[np.triu_indices_from(mask)] = True

plt.figure(figsize=(11, 9))
sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    center=0,
    linewidths=0.5,
    cbar_kws={'label': 'Coeficiente de correlación'}
)
plt.title('Matriz de Correlación — Variables Numéricas', fontsize=14, pad=20)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

### Interpretación guiada del heatmap

Ahora que vemos la matriz completa, vamos a leerla juntos.

**Ejercicio en clase:** busquen en el heatmap:
1. La correlación más fuerte (positiva)
2. La correlación más fuerte (negativa)
3. Una correlación que **los sorprenda** (diferente a lo que predijeron)
4. Una correlación que **confirme** lo esperado

**Pregunta de negocio:** si fueran un fondo de VC y solo pudieran mirar 2 variables para evaluar una startup, ¿cuáles elegirían basándose en las correlaciones? ¿Por qué?

In [ ]:
# Top correlaciones (excluyendo la diagonal)
corr_flat = corr.unstack()
corr_flat = corr_flat[corr_flat < 1.0]  # excluir autocorrelaciones
corr_flat = corr_flat[corr_flat.index.get_level_values(0) < corr_flat.index.get_level_values(1)]  # un solo triángulo

print("Top 5 correlaciones POSITIVAS:")
print(corr_flat.sort_values(ascending=False).head(5).round(3))
print()
print("Top 5 correlaciones NEGATIVAS:")
print(corr_flat.sort_values(ascending=True).head(5).round(3))

---
# Parte 4: Análisis grupal con `.groupby()` y `.agg()`

Antes de cerrar, combinamos lo aprendido en pandas para generar resúmenes por grupo que complementan los gráficos.

### `.groupby()` + `.agg()` — resumen completo por etapa

`.agg()` permite calcular **varias estadísticas al mismo tiempo** para una o varias columnas.

**Pregunta:** ¿cómo extenderían este código para agregar el churn promedio por etapa?

In [ ]:
resumen_etapa = df.groupby('etapa').agg(
    startups=('startup', 'count'),
    arr_mediana=('arr_millones', 'median'),
    arr_max=('arr_millones', 'max'),
    fondeo_promedio=('fondeo_usd_mm', 'mean'),
    nps_promedio=('nps_score', 'mean'),
    churn_promedio=('churn_mensual_pct', 'mean')
).round(2)

orden_etapa = ['Pre-seed', 'Seed', 'Serie A', 'Serie B']
resumen_etapa.loc[orden_etapa]

---
# Cierre: Resumen ejecutivo del dataset

Combinamos todo para cerrar con los números más importantes.

In [ ]:
print("=" * 55)
print("         RESUMEN DEL DATASET DE STARTUPS LATAM")
print("=" * 55)
print(f"  Total de startups analizadas : {len(df)}")
print(f"  Países representados         : {df['pais'].nunique()}")
print(f"  Sectores cubiertos           : {df['sector_principal'].nunique()}")
print(f"  Unicornios en el dataset     : {df['unicornio'].sum()}")
print()

resumen = df[['arr_millones', 'fondeo_usd_mm', 'churn_mensual_pct', 'nps_score', 'runway_meses']].agg(['mean', 'median', 'max']).round(2)
resumen.index = ['Promedio', 'Mediana', 'Máximo']
resumen

---
## Preguntas de cierre

1. ¿Qué diferencia hay entre un boxplot y un histograma? ¿Cuándo usarían cada uno?
2. ¿Qué diferencia hay entre `.describe()` y `.agg()`?
3. Nombren dos correlaciones del heatmap y expliquen una posible razón causal (o por qué podría ser espuria).
4. ¿Qué visualización agregarían para responder: *¿las startups unicornio tienen más empleados o más fondeo que las demás?*
5. Si tuvieran que predecir qué startups tienen más chances de convertirse en unicornios, ¿qué variables del dataset usarían? ¿Esto los lleva a pensar en qué tipo de modelo de ML?

---
**Próximos pasos sugeridos:** limpieza de nulos, encoding de variables categóricas, y modelos de clasificación (logistic regression, random forest) sobre la variable `unicornio`.

*Dataset: Startups LATAM — 150 empresas | Variables: sector, país, etapa, ARR, churn, NPS, fondeo, runway*